### `Generate Key`

In [ ]:
import random
import math


def generate_key() -> dict:
    while True:
        matrix = [[random.randint(0, 25) for _ in range(2)] for _ in range(2)]
        det = (matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]) % 26
        if math.gcd(det, 26) == 1:
            return {"size": 2, "matrix": matrix}


key = generate_key()
print(f"Generated Key: {key}")

### `Encryption`

In [ ]:
import os
import ast


def multiply_matrix_vector(matrix: list[list[int]], vector: list[int]) -> list[int]:
    return [
        (matrix[0][0] * vector[0] + matrix[0][1] * vector[1]) % 26,
        (matrix[1][0] * vector[0] + matrix[1][1] * vector[1]) % 26,
    ]


def encrypt(text: str, key: dict) -> str:
    matrix = key["matrix"]

    alpha_chars = [ch for ch in text if ch.isascii() and ch.isalpha()]
    clean_nums = [ord(ch.upper()) - 65 for ch in alpha_chars]

    if len(clean_nums) % 2 != 0:
        clean_nums.append(23)

    encrypted_alpha = []
    for i in range(0, len(clean_nums), 2):
        block = clean_nums[i : i + 2]
        encrypted_block = multiply_matrix_vector(matrix, block)
        encrypted_alpha.extend(chr(num + 65) for num in encrypted_block)

    result = []
    alpha_idx = 0
    for ch in text:
        if ch.isascii() and ch.isalpha():
            if alpha_idx < len(encrypted_alpha):
                result.append(encrypted_alpha[alpha_idx])
                alpha_idx += 1
            else:
                result.append(ch)
        else:
            result.append(ch)

    while alpha_idx < len(encrypted_alpha):
        result.append(encrypted_alpha[alpha_idx])
        alpha_idx += 1

    return "".join(result)


file_path = input("Enter path of plain text file: ").strip()
key_input = input("Enter key value (as dictionary string): ").strip()
key = ast.literal_eval(key_input)

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    with open(file_path, "r", encoding="utf-8") as f:
        plain_text = f.read()

    encrypted_text = encrypt(plain_text, key)

    file_name = os.path.splitext(os.path.basename(file_path))[0]
    directory = os.path.dirname(file_path)
    output_file = os.path.join(directory, f"{file_name}_Encrypted_HC.txt")

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(encrypted_text)

    print(f"Encryption successful!")
    print(f"Key used: {key}")
    print(f"Original file: {file_path}")
    print(f"Encrypted file saved: {output_file}")

### `Decryption`

In [ ]:
import os
import ast


def get_inverse_matrix(matrix: list[list[int]]) -> list[list[int]]:
    det = (matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]) % 26
    det_inv = pow(det, -1, 26)
    return [
        [(matrix[1][1] * det_inv) % 26, (-matrix[0][1] * det_inv) % 26],
        [(-matrix[1][0] * det_inv) % 26, (matrix[0][0] * det_inv) % 26],
    ]


def decrypt(text: str, key: dict) -> str:
    matrix = key["matrix"]
    inv_matrix = get_inverse_matrix(matrix)

    alpha_chars = [ch for ch in text if ch.isascii() and ch.isalpha()]
    cipher_nums = [ord(ch.upper()) - 65 for ch in alpha_chars]

    decrypted_alpha = []
    for i in range(0, len(cipher_nums), 2):
        block = cipher_nums[i : i + 2]
        decrypted_block = [
            (inv_matrix[0][0] * block[0] + inv_matrix[0][1] * block[1]) % 26,
            (inv_matrix[1][0] * block[0] + inv_matrix[1][1] * block[1]) % 26,
        ]
        decrypted_alpha.extend(chr(num + 65) for num in decrypted_block)

    result = []
    alpha_idx = 0
    for ch in text:
        if ch.isascii() and ch.isalpha():
            if alpha_idx < len(decrypted_alpha):
                result.append(decrypted_alpha[alpha_idx])
                alpha_idx += 1
            else:
                result.append(ch)
        else:
            result.append(ch)

    return "".join(result)


file_path = input("Enter path of encrypted text file: ").strip()
key_input = input("Enter key value (as dictionary string): ").strip()
key = ast.literal_eval(key_input)

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    with open(file_path, "r", encoding="utf-8") as f:
        encrypted_text = f.read()

    decrypted_text = decrypt(encrypted_text, key)

    file_name = os.path.splitext(os.path.basename(file_path))[0]
    directory = os.path.dirname(file_path)
    output_file = os.path.join(directory, f"{file_name}_Decrypted_HC.txt")

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(decrypted_text)

    print(f"Decryption successful!")
    print(f"Key used: {key}")
    print(f"Encrypted file: {file_path}")
    print(f"Decrypted file saved: {output_file}")

### `Frequency Analysis Attack`

In [ ]:
import os
import math
import itertools
from collections import Counter

ENGLISH_FREQ = {
    "A": 8.167, "B": 1.492, "C": 2.782, "D": 4.253, "E": 12.702,
    "F": 2.228, "G": 2.015, "H": 6.094, "I": 6.966, "J": 0.153,
    "K": 0.772, "L": 4.025, "M": 2.406, "N": 6.749, "O": 7.507,
    "P": 1.929, "Q": 0.095, "R": 5.987, "S": 6.327, "T": 9.056,
    "U": 2.758, "V": 0.978, "W": 2.360,
    "X": 0.150, "Y": 1.974, "Z": 0.074,
}

COMMON_BIGRAMS = {
    "TH", "HE", "IN", "ER", "AN", "RE",
    "ON", "EN", "AT", "ND", "TI", "ES",
    "OR", "TE", "OF", "ED", "IS", "IT",
    "AL", "AR", "ST", "TO", "NT", "NG",
    "SE", "HA", "AS", "OU", "IO", "LE",
}

COMMON_TRIGRAMS = {
    "THE",  "AND",  "ING",  "HER",  "HAT",  "HIS",
    "THA",  "ERE",  "FOR",  "ENT",  "ION",  "TER",
    "WAS",  "YOU",  "ITH",  "VER",  "ALL",  "WIT",
}


def chi_squared(text: str) -> float:
    n = len(text)
    if n == 0:
        return float("inf")
    count = Counter(text)
    return sum(
        ((count.get(c, 0) - (ENGLISH_FREQ[c] / 100 * n)) ** 2)
        / (ENGLISH_FREQ[c] / 100 * n)
        for c in ENGLISH_FREQ
    )


def score_text(text: str) -> float:
    if not text:
        return -1e9

    chi = chi_squared(text)
    s = -chi

    for i in range(len(text) - 1):
        if text[i : i + 2] in COMMON_BIGRAMS:
            s += 3

    for i in range(len(text) - 2):
        if text[i : i + 3] in COMMON_TRIGRAMS:
            s += 5

    return s


def get_inverse_matrix(matrix: list[list[int]]) -> list[list[int]]:
    det = (matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]) % 26
    det_inv = pow(det, -1, 26)
    return [
        [(matrix[1][1] * det_inv) % 26, (-matrix[0][1] * det_inv) % 26],
        [(-matrix[1][0] * det_inv) % 26, (matrix[0][0] * det_inv) % 26],
    ]


def hill_decrypt(text: str, key: dict) -> str:
    matrix = key["matrix"]
    inv_matrix = get_inverse_matrix(matrix)

    alpha_chars = [ch for ch in text if ch.isascii() and ch.isalpha()]
    cipher_nums = [ord(ch.upper()) - 65 for ch in alpha_chars]

    decrypted_alpha = []
    for i in range(0, len(cipher_nums), 2):
        block = cipher_nums[i : i + 2]
        decrypted_block = [
            (inv_matrix[0][0] * block[0] + inv_matrix[0][1] * block[1]) % 26,
            (inv_matrix[1][0] * block[0] + inv_matrix[1][1] * block[1]) % 26,
        ]
        decrypted_alpha.extend(chr(num + 65) for num in decrypted_block)

    result = []
    alpha_idx = 0
    for ch in text:
        if ch.isascii() and ch.isalpha():
            if alpha_idx < len(decrypted_alpha):
                result.append(decrypted_alpha[alpha_idx])
                alpha_idx += 1
            else:
                result.append(ch)
        else:
            result.append(ch)

    return "".join(result)


def hill_attack(ciphertext: str) -> dict:
    cipher_nums = [
        ord(ch) - 65 for ch in ciphertext.upper() if ch.isascii() and ch.isalpha()
    ]
    sample_len = min(200, len(cipher_nums))

    best_score = -1e9
    best_key = None

    for a, b, c, d in itertools.product(range(26), repeat=4):
        det = (a * d - b * c) % 26
        if math.gcd(det, 26) != 1:
            continue

        det_inv = pow(det, -1, 26)
        inv = [
            [(d * det_inv) % 26, (-b * det_inv) % 26],
            [(-c * det_inv) % 26, (a * det_inv) % 26],
        ]

        test_text = ""
        for i in range(0, sample_len - 1, 2):
            p0 = (inv[0][0] * cipher_nums[i] + inv[0][1] * cipher_nums[i + 1]) % 26
            p1 = (inv[1][0] * cipher_nums[i] + inv[1][1] * cipher_nums[i + 1]) % 26
            test_text += chr(p0 + 65) + chr(p1 + 65)

        score = score_text(test_text)

        if score > best_score:
            best_score = score
            best_key = [[a, b], [c, d]]

    if best_key:
        key_data = {"size": 2, "matrix": best_key}
        full_plaintext = hill_decrypt(ciphertext, key_data)

        return {
            "guessed_key": key_data,
            "guessed_plaintext": full_plaintext,
        }

    return {
        "guessed_key": None,
        "guessed_plaintext": None,
    }


file_path = input("Enter path of encrypted text file: ").strip()

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    with open(file_path, "r", encoding="utf-8") as f:
        encrypted_text = f.read()

    attack_result = hill_attack(encrypted_text)

    if attack_result["guessed_key"] is None:
        print("Attack failed to find a valid key.")
    else:
        file_name = os.path.splitext(os.path.basename(file_path))[0]
        directory = os.path.dirname(file_path)
        output_file = os.path.join(directory, f"{file_name}_Attacked_HC.txt")

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(attack_result["guessed_plaintext"])

        print(f"Attack successful!")
        print(f"Guessed Key: {attack_result['guessed_key']}")
        print(f"Output written to: {output_file}")